# Organize Xenium ouptut files for GEO
GEO requires specific files form the Xenium ouput bundle. This notebook extracts them and adds a sample identifying prefix.

In [4]:
import sys
import os
import shutil
import re
import csv
from pathlib import Path

In [6]:
sys.path.append(str(Path.cwd().resolve().parents[1]))
from config.paths import DATA_DIR
source_dir = DATA_DIR
dest_dir = DATA_DIR / "geo_organized"
raw_dir = dest_dir / "raw"
processed_dir = dest_dir / "processed"
log_file = dest_dir / "organization_log.csv"
print(dest_dir)

/home/workspace/data/temp/DRG/geo_organized


In [4]:
for d in [dest_dir, raw_dir, processed_dir]:
    d.mkdir(parents=True, exist_ok=True)

In [5]:
# GEO file definitions
raw_files = ["morphology.ome.tif", "transcripts.parquet"]
processed_files = [
    "cell_feature_matrix.h5",
    "cells.parquet",
    "cell_boundaries.parquet",
    "nucleus_boundaries.parquet"
]

In [6]:
# Extract TIS ID from output folder name
def extract_tis(folder_name):
    match = re.search(r"(TMA\d{5})", folder_name) # TMA for microarray, TIS for regular
    return match.group(1) if match else None

In [7]:
# prepare logging
log_entries = []
summary = []

In [8]:
for folder in sorted(source_dir.glob("output-*")):
    if not folder.is_dir():
        continue

    tis_id = extract_tis(folder.name)
    if not tis_id:
        print(f"Skipping {folder} (no TIS ID found)")
        continue

    sample_summary = {"Sample": tis_id, "Raw": 0, "Processed": 0, "Missing": []}

    for category, file_list, dest_dir_path in [
        ("raw", raw_files, raw_dir),
        ("processed", processed_files, processed_dir)
    ]:
        for fname in file_list:
            src = folder / fname
            dest = dest_dir_path / f"{tis_id}_{src.name}"
            if src.exists():
                shutil.move(src, dest)
                log_entries.append([tis_id, category, str(src), str(dest)])
                sample_summary[category.capitalize()] += 1
            else:
                sample_summary["Missing"].append(fname)

    summary.append(sample_summary)

In [9]:
# write log file
with open(log_file, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Sample", "Category", "Source_Path", "Destination_Path"])
    writer.writerows(log_entries)

In [10]:
# summary table
print("\nFile Organization Summary:")
print("=" * 80)
header = f"{'Sample':<10} {'Raw Files':<10} {'Processed Files':<16} Missing"
print(header)
print("-" * 80)

for s in summary:
    missing = ", ".join(s["Missing"]) if s["Missing"] else "None"
    print(f"{s['Sample']:<10} {s['Raw']:<10} {s['Processed']:<16} {missing}")

print("=" * 80)
print(f"\nDetailed log written to: {log_file}")
print(f"Raw files copied to: {raw_dir}")
print(f"Processed files copied to: {processed_dir}\n")


File Organization Summary:
Sample     Raw Files  Processed Files  Missing
--------------------------------------------------------------------------------
TMA00309   2          4                None
TMA00313   2          4                None
TMA00310   2          4                None
TMA00314   2          4                None
TMA00307   2          4                None
TMA00311   2          4                None
TMA00303   2          4                None
TMA00305   2          4                None
TMA00304   2          4                None
TMA00306   2          4                None
TMA00308   2          4                None
TMA00312   2          4                None

Detailed log written to: /home/workspace/temp/geo_organized/organization_log.csv
Raw files copied to: /home/workspace/temp/geo_organized/raw
Processed files copied to: /home/workspace/temp/geo_organized/processed

